## Lora fine-tuning

To make experimental pipeline, doing simply fine-tuning
after that, saved to local env


required: colab-pro environment

In [3]:
!nvidia-smi

Thu Apr 30 23:28:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.58.03              Driver Version: 595.58.03      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          Off |   00000000:65:00.0 Off |                    0 |
| N/A   39C    P0             48W /  300W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import  torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_id = "Qwen/Qwen2.5-1.5B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 338/338 [00:05<00:00, 63.29it/s]


In [5]:
# check the model size and vocab size
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
print(f"Model size: {model.num_parameters() / 1e9:.2f}B parameters")

Tokenizer vocab size: 151643
Model size: 1.54B parameters


In [4]:
# model anatomy
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [5]:
# chenck the first layer's self-attention q_proj weight
model.model.layers[0].self_attn.q_proj.weight

Parameter containing:
tensor([[ 0.0131, -0.0091,  0.0015,  ...,  0.0093,  0.0023,  0.0109],
        [ 0.0009, -0.0022, -0.0194,  ..., -0.0189,  0.0047, -0.0079],
        [ 0.0048,  0.0188, -0.0040,  ..., -0.0102,  0.0092,  0.0118],
        ...,
        [-0.0057, -0.0151, -0.0003,  ...,  0.0016, -0.0060, -0.0308],
        [-0.0063,  0.0137, -0.0100,  ...,  0.0225, -0.0044, -0.0090],
        [ 0.0088, -0.0203,  0.0071,  ...,  0.0178,  0.0071,  0.0157]],
       device='cuda:0', dtype=torch.bfloat16, requires_grad=True)

reference: https://huggingface.co/datasets/rajpurkar/squad_v2

In [6]:
from datasets import load_dataset

# question and answer parir dataset
dataset = load_dataset('squad_v2')

Generating validation split: 100%|██████████| 11873/11873 [00:00<00:00, 1016762.04 examples/s]


In [7]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 130319
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})

In [8]:
train_df = dataset['train']
test_df = dataset['validation']

train_df = train_df.filter(lambda x: len(x["answers"]["text"]) > 0)
test_df = test_df.filter(lambda x: len(x["answers"]["text"]) > 0)

Filter: 100%|██████████| 11873/11873 [00:00<00:00, 42249.14 examples/s]


In [9]:
def preprocess_function(example):
    answer_text = example["answers"]["text"][0]

    prompt = (
        f"### Context:\n{example['context']}\n\n"
        f"### Question:\n{example['question']}\n\n"
        "### Answer:\n"
    )
    answer = answer_text + tokenizer.eos_token
    full_text = prompt + answer

    full_tokens = tokenizer(full_text, truncation=True, max_length=768)
    prompt_tokens = tokenizer(prompt, truncation=True, max_length=768)

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]

    labels = input_ids.copy()
    prompt_len = len(prompt_tokens["input_ids"])
    labels[:prompt_len] = [-100] * prompt_len

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

In [10]:
tokenized_dataset = train_df.map(preprocess_function, batched=False)

Map: 100%|██████████| 86821/86821 [01:24<00:00, 1027.73 examples/s]


In [11]:
tokenized_dataset_test = test_df.map(preprocess_function, batched=False)

Map: 100%|██████████| 5928/5928 [00:06<00:00, 944.94 examples/s] 


In [12]:
tokenized_dataset.column_names

['id',
 'title',
 'context',
 'question',
 'answers',
 'input_ids',
 'attention_mask',
 'labels']

In [13]:
tokenized_dataset = tokenized_dataset.select_columns(["input_ids", "attention_mask", "labels"])
tokenized_dataset_test = tokenized_dataset_test.select_columns(["input_ids", "attention_mask", "labels"])

In [14]:
# select a sample from the tokenized dataset
sample = tokenized_dataset.shuffle(seed=42).select(range(2000))
sample_test = tokenized_dataset_test.shuffle(seed=42).select(range(1000))

In [15]:
from transformers import (
    Trainer, 
    TrainingArguments,
    DataCollatorForSeq2Seq
)

from peft import LoraConfig, get_peft_model, TaskType


lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)


lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()

data_collator = DataCollatorForSeq2Seq(tokenizer, padding=True, return_tensors="pt")

training_args = TrainingArguments(
    output_dir="./outputs/lora_rank=8_3_25",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    learning_rate=2e-5,
    logging_steps=20,
    save_steps=200,
    eval_steps=20,
    eval_strategy="steps",
    save_total_limit=2,
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=sample,
    eval_dataset=sample_test,
    data_collator=data_collator,
)


trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [16]:
trainer.train()

Step,Training Loss,Validation Loss
20,0.794538,nan
40,0.486155,nan
60,0.363208,nan
80,0.294474,nan
100,0.366710,nan
120,0.312399,nan
125,0.312399,nan


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.22s/it]


TrainOutput(global_step=125, training_loss=0.43406668853759767, metrics={'train_runtime': 583.6778, 'train_samples_per_second': 3.427, 'train_steps_per_second': 0.214, 'total_flos': 3026864634992640.0, 'train_loss': 0.43406668853759767, 'epoch': 1.0})

In [15]:
trainer.save_model("./outputs/lora_rank=8_3_25")
tokenizer.save_pretrained("./outputs/lora_rank=8_3_25")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./outputs/lora_rank=8_3_25/tokenizer_config.json',
 './outputs/lora_rank=8_3_25/chat_template.jinja',
 './outputs/lora_rank=8_3_25/tokenizer.json')

In [22]:
from huggingface_hub import notebook_login
notebook_login()

In [24]:
from huggingface_hub import whoami
print(whoami())

{'type': 'user', 'id': '69487a0b774219e1667e181c', 'name': 'fumi1022', 'fullname': 'fumiya nagatomo', 'email': 'fnagatomo@csuchico.edu', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1775001600, 'isPro': False, 'avatarUrl': '/avatars/f5936b25a5f369462cdd70dccfd2db46.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'fumiyasan', 'role': 'write', 'createdAt': '2026-03-26T02:35:42.010Z'}}}


In [26]:
lora_model.push_to_hub("fumi1022/lora_qwen2.5_rank8_3_25")
tokenizer.push_to_hub("fumi1022/lora_qwen2.5_rank8_3_25")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  13%|#2        | 1.11MB / 8.75MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp7g5ebn91/tokenizer.json:   0%|          | 27.7kB / 11.4MB            

CommitInfo(commit_url='https://huggingface.co/fumi1022/lora_qwen2.5_rank8_3_25/commit/8b1fb46ccf3b0a8d449eaf806c38d067f05cadae', commit_message='Upload tokenizer', commit_description='', oid='8b1fb46ccf3b0a8d449eaf806c38d067f05cadae', pr_url=None, repo_url=RepoUrl('https://huggingface.co/fumi1022/lora_qwen2.5_rank8_3_25', endpoint='https://huggingface.co', repo_type='model', repo_id='fumi1022/lora_qwen2.5_rank8_3_25'), pr_revision=None, pr_num=None)

##
lora model push up to my huggingface account